# r/AskScience ICAI review and validation — 5/10/20 constitutions

This notebook follows the 10,000-pair discovery run. It validates the selected principles on 5,000 held-out preference pairs and builds separate constitutions with 5, 10, and 20 principles.

The expensive parts are shared across batches when possible, and the same plotting code is reused for the three constitution sizes.


In [ ]:
!pip -q install openai sentence-transformers python-docx pandas numpy scipy scikit-learn matplotlib tqdm openpyxl


In [ ]:
import getpass, json, math, random, re, time
from pathlib import Path
from collections import Counter

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import Inches
from openai import AzureOpenAI
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

AUDIT_DOCX_PATH = Path(
    '/content/askscience_icai_outputs_10k_5k_cost_safe/'
    'AskScience_ICAI_principle_audit.docx'
)

if (
    not AUDIT_DOCX_PATH.exists()
    and Path('/content/AskScience_ICAI_principle_audit.docx').exists()
):
    AUDIT_DOCX_PATH = Path(
        '/content/AskScience_ICAI_principle_audit.docx'
    )

HELDOUT_CSV_PATH = Path(
    '/content/askscience heldout preference pairs.csv'
)

INITIAL_PAIR_IDS_PATH = Path(
    '/content/askscience_icai_outputs_10k_5k_cost_safe/'
    'initial_10000_pairs_used.csv'
)

# this folder keeps results from the older run from getting reused by accident.
OUTPUT_DIR = Path(
    '/content/askscience_validation_outputs_5_10_20_cost_safe'
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AZURE_ENDPOINT = (
    'https://tpicc-mj8qs589-eastus2.cognitiveservices.azure.com/'
)
AZURE_API_VERSION = '2024-12-01-preview'
AZURE_DEPLOYMENT = 'gpt-5.1'
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

# this keeps paid calls off unless i turn them on myself.
RUN_PAID_LLM = True

MAX_POST_CHARS = 1800
HELDOUT_PAIR_TARGET = 5000
SELECT_N_PRINCIPLES = 20
FINAL_CONSTITUTION_SIZES = [5, 10, 20]
MIN_APPLICABILITY = .10

# larger shared batches keep the number of requests down.
PAIR_BATCH_SIZE = 100
CONSTITUTION_BATCH_SIZE = 75
MAX_RETRIES = 2

# this estimates the full fresh-run call count.
PROJECTED_REVIEW_CALLS = 1
PROJECTED_SELECTED_TEST_CALLS = math.ceil(
    HELDOUT_PAIR_TARGET / PAIR_BATCH_SIZE
)
PROJECTED_CONSTITUTION_GENERATION_CALLS = 1
PROJECTED_CONSTITUTION_TEST_CALLS = math.ceil(
    HELDOUT_PAIR_TARGET / CONSTITUTION_BATCH_SIZE
)
PROJECTED_RULES_CALLS = 1
PROJECTED_ERROR_CALLS = 1

PROJECTED_MAX_CALLS = (
    PROJECTED_REVIEW_CALLS
    + PROJECTED_SELECTED_TEST_CALLS
    + PROJECTED_CONSTITUTION_GENERATION_CALLS
    + PROJECTED_CONSTITUTION_TEST_CALLS
    + PROJECTED_RULES_CALLS
    + PROJECTED_ERROR_CALLS
)

MAX_TOTAL_LLM_CALLS = 130  # this leaves 9 calls of headroom.

FORCE_REVIEW_SELECTION = False
FORCE_REEMBED = False
FORCE_PRINCIPLE_TEST = False
FORCE_BASELINES = False
FORCE_CONSTITUTION = False
FORCE_CONSTITUTION_TEST = False
FORCE_RULES_COMPARISON = False
FORCE_ERROR_ANALYSIS = False

OFFICIAL_RULES = [
    'Medical advice: Asking for or giving medical advice is strictly prohibited.',
    'Offensive or abusive language: Maintain professionalism; abusive/offensive language is not allowed outside research context.',
    'Homework question: Homework questions should not be answered by AskScience experts.',
    'Meme, joke, or just link: Memes, jokes, or comments consisting solely of a link are not allowed.',
    'Top-level comment not answering the question: Direct replies must answer the question or ask a follow-up question.',
    'Nonreputable answer: Answers should be supported by reputable sources and scientific research; do not cite yourself.',
    'Simple calculation or easily Googleable: Questions answerable by a simple Google search or simple calculation are not appropriate.',
    'No bots, no AI generated answers.'
]

client = None

if RUN_PAID_LLM:
    subscription_key = getpass.getpass('Azure OpenAI API key: ')

    client = AzureOpenAI(
        api_version=AZURE_API_VERSION,
        azure_endpoint=AZURE_ENDPOINT,
        api_key=subscription_key
    )

print(
    f'Held-out target: {HELDOUT_PAIR_TARGET:,} completely new pairs; '
    f'final constitution sizes: {FINAL_CONSTITUTION_SIZES}.'
)
print('\nPAID-CALL ESTIMATE (full fresh run, before checkpoints):')
print(f'  Review/select 20 candidates: {PROJECTED_REVIEW_CALLS}')
print(
    '  Shared 20-principle + no-constitution held-out pass: '
    f'{PROJECTED_SELECTED_TEST_CALLS}'
)
print(
    '  Generate independent 5/10/20 constitutions in ONE call: '
    f'{PROJECTED_CONSTITUTION_GENERATION_CALLS}'
)
print(
    '  Shared 5/10/20 full + constituent-principle pass: '
    f'{PROJECTED_CONSTITUTION_TEST_CALLS}'
)
print(f'  Rules comparison: {PROJECTED_RULES_CALLS}')
print(f'  Error/bias analysis: {PROJECTED_ERROR_CALLS}')
print(f'  Total planned maximum: {PROJECTED_MAX_CALLS}')
print(f'  Emergency hard cap: {MAX_TOTAL_LLM_CALLS}')
print(
    '  Note: Azure billing is token-based; call count is a '
    'safety proxy, not a dollar estimate.'
)
print(
    'PAID LLM STATUS:',
    'ENABLED' if RUN_PAID_LLM else 'BLOCKED (dry-run safe)'
)

if PROJECTED_MAX_CALLS > MAX_TOTAL_LLM_CALLS:
    raise RuntimeError(
        'Configuration error: projected call plan exceeds the hard cap.'
    )


## shared helpers and call budget


In [ ]:
CALL_LOG = OUTPUT_DIR / 'llm_call_log.jsonl'

def save_json(p, o):
    Path(p).write_text(
        json.dumps(o, indent=2, ensure_ascii=False),
        encoding='utf-8'
    )

def load_json(p):
    return json.loads(
        Path(p).read_text(encoding='utf-8')
    )

def read_jsonl(p):
    p = Path(p)

    if not p.exists():
        return []

    with p.open(encoding='utf-8') as f:
        return [json.loads(x) for x in f if x.strip()]

def append_jsonl(p, rows):
    with Path(p).open('a', encoding='utf-8') as f:
        for r in rows:
            f.write(
                json.dumps(r, ensure_ascii=False) + '\n'
            )

def calls():
    return len(read_jsonl(CALL_LOG))

def budget(stage, n):
    projected = calls() + n

    print(
        f'{stage}: about {n} new calls; '
        f'projected {projected}/{MAX_TOTAL_LLM_CALLS}'
    )

    if projected > MAX_TOTAL_LLM_CALLS:
        raise RuntimeError(
            'Projected calls exceed hard cap; no call was sent.'
        )

    if n and not RUN_PAID_LLM:
        print(
            '  DRY RUN: paid calls remain blocked. '
            'Set RUN_PAID_LLM=True only after reviewing this estimate.'
        )

def extract_json(t):
    t = (t or '').strip()
    t = re.sub(r'^```(?:json)?\s*', '', t, flags=re.I)
    t = re.sub(r'\s*```$', '', t)

    try:
        return json.loads(t)

    except Exception:
        starts = [
            i for i in [t.find('{'), t.find('[')]
            if i >= 0
        ]

        return json.loads(
            t[
                min(starts):
                max(t.rfind('}'), t.rfind(']')) + 1
            ]
        )

def ask(
    system,
    user,
    stage,
    retries=MAX_RETRIES,
    max_tokens=12000
):
    if not RUN_PAID_LLM:
        raise RuntimeError(
            'Paid LLM calls are BLOCKED. Review the printed estimate, '
            'then set RUN_PAID_LLM=True and rerun from the configuration cell.'
        )

    if client is None:
        raise RuntimeError(
            'RUN_PAID_LLM=True but Azure client is not initialized. '
            'Rerun the configuration cell.'
        )

    last = None

    for a in range(retries):
        if calls() >= MAX_TOTAL_LLM_CALLS:
            raise RuntimeError(
                'LLM call hard cap reached. '
                'No additional paid call was sent.'
            )

        try:
            r = client.chat.completions.create(
                model=AZURE_DEPLOYMENT,
                messages=[
                    {'role': 'system', 'content': system},
                    {'role': 'user', 'content': user}
                ],
                max_completion_tokens=max_tokens,
                response_format={'type': 'json_object'}
            )

            u = getattr(r, 'usage', None)

            append_jsonl(
                CALL_LOG,
                [{
                    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
                    'stage': stage,
                    'prompt_tokens': getattr(u, 'prompt_tokens', None),
                    'completion_tokens': getattr(u, 'completion_tokens', None),
                    'total_tokens': getattr(u, 'total_tokens', None)
                }]
            )

            return extract_json(r.choices[0].message.content)

        except Exception as e:
            last = e
            print('retry', a + 1, e)

            if a < retries - 1:
                time.sleep(min(20, 2 ** a * 3))

    raise RuntimeError(stage) from last

def clean(v):
    return '' if pd.isna(v) else re.sub(r'\s+', ' ', str(v)).strip()

def post_text(t, b):
    t, b = clean(t), clean(b)

    return (
        ('Title: ' + t + '\n' if t else '')
        + ('Body: ' + b if b else '')
    )

def chunks(x, n):
    for i in range(0, len(x), n):
        yield x[i:i + n]

def corr(x, y, kind='spearman'):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)

    if m.sum() < 3 or np.std(x[m]) == 0 or np.std(y[m]) == 0:
        return np.nan, np.nan

    return (
        spearmanr(x[m], y[m])
        if kind == 'spearman'
        else pearsonr(x[m], y[m])
    )

def boot(x, B=3000):
    x = np.asarray(x, float)
    r = np.random.default_rng(SEED)

    z = np.array([
        r.choice(
            x,
            len(x),
            replace=True
        ).mean()
        for _ in range(B)
    ])

    return np.quantile(z, [.025, .975])


## steps 1–2 — review the top 20 principles

The 20 principles from Notebook 1 are reviewed in one call and rewritten into 20 clearer candidates for held-out testing. I keep all 20 here so there is enough evidence to build the 20-principle constitution later.


In [ ]:
def top20(path):
    if not Path(path).exists():
        raise FileNotFoundError(
            f'Principle audit not found: {path}'
        )

    d = Document(path)
    t = d.tables[0]

    h = [
        clean(c.text)
        for c in t.rows[0].cells
    ]

    rows = [
        dict(
            zip(
                h,
                [clean(c.text) for c in r.cells]
            )
        )
        for r in t.rows[1:]
    ]

    f = pd.DataFrame(rows).head(20)

    idc = next(
        (
            x for x in f.columns
            if 'id' in x.lower()
        ),
        f.columns[0]
    )

    pc = next(
        (
            x for x in f.columns
            if 'principle' in x.lower()
        ),
        f.columns[1]
    )

    return (
        f.rename(
            columns={
                idc: 'source_id',
                pc: 'principle'
            }
        )[['source_id', 'principle']]
    )

initial_20 = top20(AUDIT_DOCX_PATH)
path = OUTPUT_DIR / 'step12_review_selection.json'

if path.exists() and not FORCE_REVIEW_SELECTION:
    res = load_json(path)

else:
    budget('Review + selection', 1)
    payload = initial_20.to_dict('records')

    user = f'''Review these 20 inferred r/AskScience principles for clarity, generalizability, redundancy, and artifact risk, then in the SAME response produce exactly {SELECT_N_PRINCIPLES} concise tested candidate values. Preserve distinct useful ideas rather than collapsing the list below 20; rewrite overlapping items so each candidate has a clear, non-duplicative scope. Every selected principle must cite source IDs. Return JSON: {{"reviews":[{{"source_id":"...","clarity":"high|medium|low","generalizability":"high|medium|low","artifact_risk":"high|medium|low","notes":"..."}}],"selected":[{{"selected_id":"S01","principle":"...","theme":"...","expected_scope":"broad|medium|narrow","source_ids":["..."]}}],"selection_summary":"..."}}. Principles: {json.dumps(payload,ensure_ascii=False)}'''

    res = ask(
        'Review inferred community principles conservatively. Return JSON only.',
        user,
        'review_selection'
    )

    save_json(path, res)

review_df = pd.DataFrame(
    res.get('reviews', [])
)

selected_df = (
    pd.DataFrame(
        res.get('selected', [])
    )
    .head(SELECT_N_PRINCIPLES)
)

if len(selected_df) != SELECT_N_PRINCIPLES:
    raise ValueError(
        f'Expected exactly {SELECT_N_PRINCIPLES} selected principles, '
        f'got {len(selected_df)}. '
        'Set FORCE_REVIEW_SELECTION=True and rerun this cell.'
    )

selected_df.to_csv(
    OUTPUT_DIR / 'selected_principles.csv',
    index=False
)

print(
    selected_df[
        ['selected_id', 'principle']
    ].to_string(index=False)
)


## load the 5,000 held-out pairs and save embeddings

This checks the held-out file, keeps exactly 5,000 usable pairs, randomizes which post appears as A or B, and saves the embeddings so they don't have to be recomputed.


In [ ]:
# this cleans the held-out pairs, randomizes a/b, and saves the embeddings once.
h = pd.read_csv(
    HELDOUT_CSV_PATH,
    keep_default_na=False
)

req = {
    'pair_id',
    'preferred_post_id',
    'preferred_post_title',
    'preferred_post_body',
    'preferred_upvote_ratio',
    'nonpreferred_post_id',
    'nonpreferred_post_title',
    'nonpreferred_post_body',
    'nonpreferred_upvote_ratio'
}

miss = req - set(h.columns)

if miss:
    raise ValueError(miss)

h['preferred_text'] = [
    post_text(t, b)[:MAX_POST_CHARS]
    for t, b in zip(
        h.preferred_post_title,
        h.preferred_post_body
    )
]

h['nonpreferred_text'] = [
    post_text(t, b)[:MAX_POST_CHARS]
    for t, b in zip(
        h.nonpreferred_post_title,
        h.nonpreferred_post_body
    )
]

h = (
    h[
        (h.preferred_text != '')
        & (h.nonpreferred_text != '')
    ]
    .copy()
    .reset_index(drop=True)
)

h['pair_id'] = h.pair_id.astype(str)

h['upvote_ratio_gap'] = pd.to_numeric(
    h.get(
        'upvote_ratio_gap',
        h.preferred_upvote_ratio - h.nonpreferred_upvote_ratio
    ),
    errors='coerce'
)

if len(h) < HELDOUT_PAIR_TARGET:
    raise ValueError(
        f'Need {HELDOUT_PAIR_TARGET:,} usable held-out pairs, '
        f'but only {len(h):,} remain after cleaning.'
    )

if len(h) > HELDOUT_PAIR_TARGET:
    h = (
        h.sample(
            n=HELDOUT_PAIR_TARGET,
            random_state=SEED
        )
        .sort_values('pair_id')
        .reset_index(drop=True)
    )

if h.pair_id.duplicated().any():
    raise ValueError(
        'Held-out pair_id values must be unique.'
    )

if INITIAL_PAIR_IDS_PATH.exists():
    initial_ids = set(
        pd.read_csv(
            INITIAL_PAIR_IDS_PATH,
            usecols=['pair_id']
        )
        .pair_id
        .astype(str)
    )

    overlap = initial_ids & set(h.pair_id)

    if overlap:
        raise ValueError(
            'Held-out data overlaps the initial 10,000 on '
            f'{len(overlap)} pair_id values. '
            'Regenerate the held-out SQL pairs.'
        )

r = random.Random(SEED)

h['preferred_position'] = [
    r.choice(['A', 'B'])
    for _ in range(len(h))
]

h['post_a'] = np.where(
    h.preferred_position.eq('A'),
    h.preferred_text,
    h.nonpreferred_text
)

h['post_b'] = np.where(
    h.preferred_position.eq('A'),
    h.nonpreferred_text,
    h.preferred_text
)

h.to_csv(
    OUTPUT_DIR / 'heldout_randomized.csv',
    index=False
)

model = SentenceTransformer(EMBED_MODEL)
ef = OUTPUT_DIR / 'heldout_selected_embeddings.npz'

if ef.exists() and not FORCE_REEMBED:
    z = np.load(ef)
    pe = z['preferred']
    ne = z['nonpreferred']
    se = z['principles']

    if len(pe) != len(h) or len(se) != len(selected_df):
        raise ValueError(
            'Saved embedding dimensions do not match this run. '
            'Set FORCE_REEMBED=True.'
        )

else:
    e = model.encode(
        (
            h.preferred_text.tolist()
            + h.nonpreferred_text.tolist()
            + selected_df.principle.tolist()
        ),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    pe = e[:len(h)]
    ne = e[len(h):2 * len(h)]
    se = e[2 * len(h):]

    np.savez_compressed(
        ef,
        preferred=pe,
        nonpreferred=ne,
        principles=se,
        pair_ids=h.pair_id.to_numpy()
    )

pair_dist = 1 - np.sum(
    pe * ne,
    axis=1
)

h['embedding_distance'] = pair_dist

pr, pp = corr(
    pair_dist,
    h.upvote_ratio_gap,
    'pearson'
)

sr, sp = corr(
    pair_dist,
    h.upvote_ratio_gap,
    'spearman'
)

pd.DataFrame([{
    'pearson_r': pr,
    'pearson_p': pp,
    'spearman_rho': sr,
    'spearman_p': sp
}]).to_csv(
    OUTPUT_DIR / 'embedding_distance_upvote_gap_correlation.csv',
    index=False
)

plt.figure(figsize=(7, 5))
plt.scatter(
    pair_dist,
    h.upvote_ratio_gap,
    alpha=.4
)
plt.xlabel('Pair cosine distance')
plt.ylabel('Upvote-ratio gap')
plt.title(
    f'Semantic distance vs preference gap (rho={sr:.3f})'
)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / 'embedding_distance_vs_upvote_gap.png',
    dpi=180
)
plt.close()

# this uses embedding margins as a cheap comparison before the llm evaluation.
spref = se @ pe.T
snon = se @ ne.T

pol = np.where(
    selected_df.principle
    .str.lower()
    .str.startswith('avoid'),
    -1.,
    1.
)[:, None]

margin = pol * (spref - snon)
em = []

for i, row in selected_df.reset_index(drop=True).iterrows():
    rho, p = corr(
        margin[i],
        h.upvote_ratio_gap
    )

    em.append({
        'selected_id': row.selected_id,
        'embedding_accuracy': float(
            (margin[i] > 0).mean()
        ),
        'mean_embedding_margin': float(
            margin[i].mean()
        ),
        'margin_vs_gap_spearman': rho,
        'margin_vs_gap_p': p
    })

embedding_metrics = pd.DataFrame(em)

embedding_metrics.to_csv(
    OUTPUT_DIR / 'embedding_principle_metrics.csv',
    index=False
)

print('Embeddings saved once:', ef)
print('Held-out pairs:', len(h))


In [ ]:

# held-out csv diagnostic / safe load
# zero paid llm calls

import pandas as pd

RUN_PAID_LLM = False

# important:
# keep_default_na=false prevents pandas from silently converting
# text-looking values into nan.
h = pd.read_csv(
    HELDOUT_CSV_PATH,
    keep_default_na=False
)

req = {
    'pair_id',
    'preferred_post_id',
    'preferred_post_title',
    'preferred_post_body',
    'preferred_upvote_ratio',
    'nonpreferred_post_id',
    'nonpreferred_post_title',
    'nonpreferred_post_body',
    'nonpreferred_upvote_ratio'
}

miss = req - set(h.columns)

if miss:
    raise ValueError(
        f'Missing required columns: {sorted(miss)}'
    )

print("========== HELD-OUT RAW CSV DIAGNOSTIC ==========")
print(f'Rows loaded from CSV: {len(h):,}')
print(f'Unique pair_id values: {h.pair_id.astype(str).nunique():,}')

if 'available_new_pair_pool_size' in h.columns:
    vals = pd.to_numeric(
        h['available_new_pair_pool_size'],
        errors='coerce'
    ).dropna()

    if len(vals):
        print(
            f'Available new BigQuery pool size: '
            f'{int(vals.max()):,}'
        )

print()

for side in ['preferred', 'nonpreferred']:

    title_col = f'{side}_post_title'
    body_col = f'{side}_post_body'

    blank_title = (
        h[title_col]
        .astype(str)
        .str.strip()
        .eq('')
    )

    blank_body = (
        h[body_col]
        .astype(str)
        .str.strip()
        .eq('')
    )

    print(
        f'{side.capitalize()} blank title: '
        f'{blank_title.sum():,}'
    )

    print(
        f'{side.capitalize()} blank body:  '
        f'{blank_body.sum():,}'
    )

    print(
        f'{side.capitalize()} BOTH blank:  '
        f'{(blank_title & blank_body).sum():,}'
    )

print("=================================================")

# build the exact text notebook 2 uses

h['preferred_text'] = [
    post_text(t, b)[:MAX_POST_CHARS]
    for t, b in zip(
        h.preferred_post_title,
        h.preferred_post_body
    )
]

h['nonpreferred_text'] = [
    post_text(t, b)[:MAX_POST_CHARS]
    for t, b in zip(
        h.nonpreferred_post_title,
        h.nonpreferred_post_body
    )
]

usable = (
    h['preferred_text'].ne('')
    &
    h['nonpreferred_text'].ne('')
)

print()
print("========== NOTEBOOK-2 CLEANING CHECK ==========")
print(f'Raw pairs:          {len(h):,}')
print(f'Usable pairs:       {usable.sum():,}')
print(f'Would be removed:   {(~usable).sum():,}')
print(
    'PAID LLM CALLS MADE: 0'
)
print("===============================================")


## step 3 — test the selected principles and the no-constitution baseline

Each held-out pair is sent once per batch. The model gives a no-constitution A/B choice and a separate A/B/None vote for every selected principle. Missing principle votes are treated as an error instead of being silently counted as `None`.


In [ ]:
RUN_PAID_LLM=True


In [ ]:

# fix held-out a/b randomization columns
# this doesn't use any llm calls.

import random
import numpy as np

RUN_PAID_LLM = False

# make sure ids are strings
h['pair_id'] = h['pair_id'].astype(str)

# re-create preferred/nonpreferred text if needed
if 'preferred_text' not in h.columns:
    h['preferred_text'] = [
        post_text(t, b)[:MAX_POST_CHARS]
        for t, b in zip(
            h['preferred_post_title'],
            h['preferred_post_body']
        )
    ]

if 'nonpreferred_text' not in h.columns:
    h['nonpreferred_text'] = [
        post_text(t, b)[:MAX_POST_CHARS]
        for t, b in zip(
            h['nonpreferred_post_title'],
            h['nonpreferred_post_body']
        )
    ]

# deterministically randomize which side contains the preferred post
r = random.Random(SEED)

h['preferred_position'] = [
    r.choice(['A', 'B'])
    for _ in range(len(h))
]

# construct the text shown to the llm
h['post_a'] = np.where(
    h['preferred_position'].eq('A'),
    h['preferred_text'],
    h['nonpreferred_text']
)

h['post_b'] = np.where(
    h['preferred_position'].eq('A'),
    h['nonpreferred_text'],
    h['preferred_text']
)

# save the randomized version
h.to_csv(
    OUTPUT_DIR / 'heldout_randomized.csv',
    index=False
)

print("========== HELD-OUT A/B FIX ==========")
print(f"Pairs: {len(h):,}")
print("Columns created:")
print("  preferred_position:", 'preferred_position' in h.columns)
print("  post_a:", 'post_a' in h.columns)
print("  post_b:", 'post_b' in h.columns)
print()
print(h[['pair_id', 'preferred_position']].head())
print()
print("PAID LLM CALLS MADE: 0")
print("======================================")


In [ ]:
# this checks every selected principle and won't fill in missing votes as "none".
ck = OUTPUT_DIR / 'step3_principle_and_baseline_votes.jsonl'

if (
    FORCE_PRINCIPLE_TEST
    or FORCE_BASELINES
) and ck.exists():
    ck.unlink()

done = {
    str(x['pair_id'])
    for x in read_jsonl(ck)
}

principles_payload = selected_df[
    ['selected_id', 'principle']
].to_dict('records')

required_sids = (
    selected_df.selected_id
    .astype(str)
    .tolist()
)

vote_schema = {
    sid: 'A|B|None'
    for sid in required_sids
}

pend = [
    {
        'pair_id': str(x.pair_id),
        'post_a': x.post_a,
        'post_b': x.post_b
    }
    for _, x in h.iterrows()
    if str(x.pair_id) not in done
]

budget(
    '20-principle full held-out test + no-constitution baseline',
    math.ceil(
        len(pend) / PAIR_BATCH_SIZE
    )
)

for batch in tqdm(
    list(
        chunks(
            pend,
            PAIR_BATCH_SIZE
        )
    ),
    desc='Selected principles + baseline'
):
    response_schema = {
        'results': [{
            'pair_id': '...',
            'no_constitution_vote': 'A|B',
            'principle_votes': vote_schema
        }]
    }

    user = f'''For every pair, do two tasks in the SAME response.

1. Predict which r/AskScience post the community would prefer with NO constitution: A or B.

2. Apply EVERY supplied principle independently to the same pair and vote A, B, or None according ONLY to that principle. None means inapplicable.

CRITICAL REQUIREMENT:
For EVERY pair you MUST return a vote for ALL of these principle IDs:
{json.dumps(required_sids)}

Do not omit any principle ID.

Return JSON in exactly this structure:
{json.dumps(response_schema)}

Principles:
{json.dumps(principles_payload,ensure_ascii=False)}

Pairs:
{json.dumps(batch,ensure_ascii=False)}
'''

    out = ask(
        (
            'Predict community preference and independently apply '
            'EVERY supplied principle. Never omit a principle ID. '
            'Return JSON only.'
        ),
        user,
        'selected_principles_plus_baseline'
    )

    got = {
        str(x.get('pair_id')): x
        for x in out.get('results', [])
        if isinstance(x, dict)
    }

    rows = []
    bad_pairs = []

    for x in batch:
        pid = x['pair_id']

        if pid not in got:
            bad_pairs.append(pid)
            continue

        g = got[pid]
        pv = g.get('principle_votes')

        if not isinstance(pv, dict):
            bad_pairs.append(pid)
            continue

        missing_ids = [
            sid
            for sid in required_sids
            if sid not in pv
        ]

        bad_ids = [
            sid
            for sid in required_sids
            if (
                sid in pv
                and str(pv[sid]) not in {'A', 'B', 'None'}
            )
        ]

        bv0 = str(
            g.get(
                'no_constitution_vote',
                ''
            )
        )

        if (
            missing_ids
            or bad_ids
            or bv0 not in {'A', 'B'}
        ):
            bad_pairs.append(pid)
            continue

        clean_votes = {
            sid: str(pv[sid])
            for sid in required_sids
        }

        rows.append({
            'pair_id': pid,
            'no_constitution_vote': bv0,
            'principle_votes': clean_votes
        })

    if rows:
        append_jsonl(
            ck,
            rows
        )

    if bad_pairs:
        raise RuntimeError(
            f'{len(bad_pairs)} pairs had incomplete Step 3 output. '
            'The valid pairs were checkpointed, so rerunning this cell '
            'will only request the missing pairs.'
        )

raw_eval = read_jsonl(ck)
truth = h.set_index('pair_id').preferred_position.to_dict()
vote_rows = []

for r0 in raw_eval:
    for sid, v in r0.get('principle_votes', {}).items():
        vote_rows.append({
            'selected_id': str(sid),
            'pair_id': str(r0['pair_id']),
            'vote': str(v)
        })

votes = pd.DataFrame(vote_rows)

votes['truth'] = votes.pair_id.map(truth)
votes['applicable'] = votes.vote.isin(['A', 'B'])
votes['correct'] = (
    votes.applicable
    & votes.vote.eq(votes.truth)
)

m = (
    votes.groupby('selected_id')
    .agg(
        tested=('pair_id', 'nunique'),
        applicable=('applicable', 'sum'),
        correct=('correct', 'sum')
    )
    .reset_index()
)

m['applicability_rate'] = (
    m.applicable
    / m.tested
)

m['accuracy_when_applicable'] = (
    m.correct
    / m.applicable.replace(0, np.nan)
)

m['net_score'] = (
    m.correct
    - (m.applicable - m.correct)
)

metrics = (
    selected_df
    .merge(
        m,
        on='selected_id',
        how='left'
    )
    .merge(
        embedding_metrics,
        on='selected_id',
        how='left'
    )
)

metrics['passes_validation'] = (
    (metrics.applicability_rate >= MIN_APPLICABILITY)
    & (metrics.net_score > 0)
)

metrics.to_csv(
    OUTPUT_DIR / 'heldout_principle_metrics.csv',
    index=False
)

selected_metrics = metrics.copy()

rel = {
    selected_df.iloc[i].selected_id:
        h.iloc[
            np.argsort(
                -np.abs(margin[i])
            )
        ].pair_id.astype(str).tolist()
    for i in range(len(selected_df))
}

bv = pd.DataFrame([
    {
        'pair_id': str(x['pair_id']),
        'vote': str(
            x.get(
                'no_constitution_vote',
                ''
            )
        )
    }
    for x in raw_eval
])

bv['truth'] = bv.pair_id.map(truth)
bv['correct'] = bv.vote.eq(bv.truth)

bv.to_csv(
    OUTPUT_DIR / 'no_constitution_votes.csv',
    index=False
)

print(
    metrics[
        [
            'selected_id',
            'tested',
            'applicability_rate',
            'accuracy_when_applicable',
            'net_score'
        ]
    ].to_string(index=False)
)

print(
    'No-constitution accuracy:',
    bv.correct.mean()
)


## principle accuracy, applicability, and stability graphs


In [ ]:
p = metrics.sort_values(
    'accuracy_when_applicable',
    ascending=False
)

x = np.arange(len(p))
w = .38

plt.figure(figsize=(10, 5))
plt.bar(
    x - w / 2,
    p.accuracy_when_applicable,
    w,
    label='Accuracy'
)
plt.bar(
    x + w / 2,
    p.applicability_rate,
    w,
    label='Applicability'
)
plt.axhline(.5, linestyle='--')
plt.xticks(x, p.selected_id)
plt.ylim(0, 1)
plt.legend()
plt.title('Principle accuracy and applicability')
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / 'principle_accuracy_and_applicability.png',
    dpi=180
)
plt.close()

# this shows whether each principle's accuracy stays stable as more pairs are added.
plt.figure(figsize=(9, 6))

for sid, g in votes.groupby('selected_id'):
    order = {
        pid: i
        for i, pid in enumerate(rel[sid])
    }

    g = g.copy()
    g['o'] = g.pair_id.map(order)
    g = g.sort_values('o')

    app = g.applicable.astype(int).cumsum()
    cor = g.correct.astype(int).cumsum()

    plt.plot(
        np.arange(1, len(g) + 1),
        cor / app.replace(0, np.nan),
        label=sid
    )

plt.axhline(.5, linestyle='--')
plt.xlabel('Number of preference pairs tested')
plt.ylabel('Cumulative accuracy when applicable')
plt.title('Principle accuracy as more posts are tested')
plt.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / 'principle_cumulative_accuracy.png',
    dpi=180
)
plt.close()

rho, pv = corr(
    metrics.embedding_accuracy,
    metrics.accuracy_when_applicable
)

plt.figure(figsize=(7, 5))
plt.scatter(
    metrics.embedding_accuracy,
    metrics.accuracy_when_applicable
)
plt.axhline(.5, linestyle='--')
plt.xlabel('Embedding-only accuracy')
plt.ylabel('LLM accuracy')
plt.title(
    f'Embedding vs LLM accuracy (rho={rho:.3f})'
)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / 'embedding_vs_llm_accuracy.png',
    dpi=180
)
plt.close()

pd.DataFrame([{
    'spearman_rho': rho,
    'p_value': pv
}]).to_csv(
    OUTPUT_DIR / 'embedding_vs_llm_accuracy_correlation.csv',
    index=False
)


## step 4 — simple baselines

The no-constitution result is reused from Step 3, so this section doesn't need another LLM pass.


In [ ]:
pos = h.preferred_position.to_numpy()
n = len(h)
bas = []

def add(
    name,
    acc,
    notes='',
    ci=(np.nan, np.nan)
):
    bas.append({
        'method': name,
        'accuracy': acc,
        'ci_low': ci[0],
        'ci_high': ci[1],
        'notes': notes
    })

r = np.random.default_rng(SEED)

add(
    'Random A/B',
    np.mean([
        np.mean(
            r.choice(
                ['A', 'B'],
                n
            ) == pos
        )
        for _ in range(1000)
    ]),
    'Chance'
)

add(
    'Always A',
    np.mean(pos == 'A'),
    'Position check'
)

add(
    'Always B',
    np.mean(pos == 'B'),
    'Position check'
)

pl = h.preferred_text.str.len().to_numpy()
nl = h.nonpreferred_text.str.len().to_numpy()

add(
    'Longer text',
    np.mean(
        np.where(
            pl > nl,
            1,
            np.where(
                pl < nl,
                0,
                .5
            )
        )
    ),
    'Surface heuristic'
)

no_acc = bv.correct.mean()
lo, hi = boot(bv.correct)

add(
    'LLM without constitution',
    no_acc,
    'Same shared Step 3 calls, no principles',
    (lo, hi)
)

best = (
    metrics.sort_values(
        'net_score',
        ascending=False
    )
    .iloc[0]
)

pvotes = votes[
    votes.selected_id == best.selected_id
]

blo, bhi = boot(pvotes.correct)

add(
    f'Best single principle ({best.selected_id})',
    pvotes.correct.mean(),
    'None counts as wrong',
    (blo, bhi)
)

baseline = pd.DataFrame(bas)

baseline.to_csv(
    OUTPUT_DIR / 'baselines.csv',
    index=False
)


## step 5 — build separate 5-, 10-, and 20-principle constitutions

All three sizes are requested in one call, but each one is supposed to be its own solution instead of just adding more rules to the smaller version.


In [ ]:
cf = OUTPUT_DIR / 'step5_constitutions_5_10_20.json'

ranked = metrics.sort_values(
    [
        'passes_validation',
        'net_score',
        'accuracy_when_applicable'
    ],
    ascending=False
)

evidence = ranked[
    [
        'selected_id',
        'principle',
        'applicability_rate',
        'accuracy_when_applicable',
        'net_score',
        'passes_validation'
    ]
].to_dict('records')

# this builds the three sizes separately even though they share one llm call.
if cf.exists() and not FORCE_CONSTITUTION:
    cres = load_json(cf)

else:
    budget(
        'Independent 5/10/20 constitution generation',
        1
    )

    user = f'''Using the tested evidence below, create THREE SEPARATE inferred r/AskScience constitutions in this ONE response: exactly 5 principles, exactly 10 principles, and exactly 20 principles. Treat each size as a fresh independent optimization problem. The 10-principle constitution must be designed as if the 5-principle constitution had never been created; do NOT merely append five more principles. The 20-principle constitution must likewise be designed independently as if neither smaller constitution existed; do NOT merely extend either smaller list. Overlap is allowed only when the same principle is independently among the strongest choices for that size. Keep principles concise, generalizable, non-redundant within each constitution, and grounded in the tested evidence. Return JSON exactly as {{"constitution_5":[{{"constitution_id":"C5_01","principle":"...","source_ids":["S01"],"evidence_rationale":"..."}}],"constitution_10":[{{"constitution_id":"C10_01","principle":"...","source_ids":["S01"],"evidence_rationale":"..."}}],"constitution_20":[{{"constitution_id":"C20_01","principle":"...","source_ids":["S01"],"evidence_rationale":"..."}}],"summary_5":"...","summary_10":"...","summary_20":"..."}}. Evidence: {json.dumps(evidence,ensure_ascii=False)}'''

    cres = ask(
        'Create three independent inferred constitutions from tested evidence. Return JSON only.',
        user,
        'constitution_generation_5_10_20'
    )

    save_json(
        cf,
        cres
    )

constitutions = {}

for size in FINAL_CONSTITUTION_SIZES:
    key = f'constitution_{size}'
    frame = pd.DataFrame(
        cres.get(
            key,
            []
        )
    )

    if len(frame) != size:
        raise ValueError(
            f'Expected exactly {size} principles in {key}, '
            f'got {len(frame)}. '
            'Set FORCE_CONSTITUTION=True and rerun this cell.'
        )

    need = {
        'constitution_id',
        'principle'
    }

    if not need.issubset(frame.columns):
        raise ValueError(
            f'{key} is missing required fields: '
            f'{sorted(need - set(frame.columns))}'
        )

    constitutions[size] = (
        frame.reset_index(drop=True)
    )

    frame.to_csv(
        OUTPUT_DIR / f'constitution_{size}.csv',
        index=False
    )

    print(
        f'\n{size}-principle constitution'
    )

    print(
        frame[
            ['constitution_id', 'principle']
        ].to_string(index=False)
    )

constitution_5 = constitutions[5]
constitution_10 = constitutions[10]
constitution_20 = constitutions[20]


## step 6 — evaluate the full constitutions and each principle

For every held-out pair, the model gives one vote for the full 5-, 10-, and 20-principle constitutions and also evaluates every individual principle. The corrected parser checks that every required vote is actually present before saving the result.


In [ ]:
# this smaller batch was used to repair missing step 6 pairs.
# smaller batches make it less likely that the json response gets cut off.

CONSTITUTION_BATCH_SIZE = 40

print("Repair batch size:", CONSTITUTION_BATCH_SIZE)
print("Existing 4,771 valid results will be reused.")


In [ ]:
# this reruns the 5/10/20 evaluation and checks every principle vote.
# missing votes aren't counted as "none"; incomplete pairs stay unfinished.

import math
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# new corrected checkpoint

ckc = (
    OUTPUT_DIR /
    'step6_constitutions_5_10_20_votes_CORRECTED.jsonl'
)

# constitution payload + required principle order

constitution_payload = {

    str(size):
        constitutions[size][
            [
                'constitution_id',
                'principle'
            ]
        ].to_dict('records')

    for size in FINAL_CONSTITUTION_SIZES
}

required_constitution_ids = {

    str(size):
        constitutions[size][
            'constitution_id'
        ]
        .astype(str)
        .tolist()

    for size in FINAL_CONSTITUTION_SIZES
}

# safety: verify ids have expected sizes.

for size in FINAL_CONSTITUTION_SIZES:

    sk = str(size)

    if len(
        required_constitution_ids[sk]
    ) != size:

        raise RuntimeError(
            f'Constitution {size} has '
            f'{len(required_constitution_ids[sk])} IDs; '
            f'expected exactly {size}.'
        )

print(
    'Required constituent principles:'
)

for size in FINAL_CONSTITUTION_SIZES:

    print(
        f'  {size}: '
        f'{required_constitution_ids[str(size)]}'
    )

# completed pairs from corrected checkpoint only

done = {

    str(x['pair_id'])

    for x in read_jsonl(
        ckc
    )

}

pend = [

    {
        'pair_id':
            str(x.pair_id),

        'post_a':
            x.post_a,

        'post_b':
            x.post_b
    }

    for _, x in h.iterrows()

    if str(
        x.pair_id
    ) not in done

]

needed_calls = math.ceil(
    len(pend) /
    CONSTITUTION_BATCH_SIZE
)

# call-cap safety for the corrected rerun

# the old 130-call cap was for the original fresh run.
# since those calls are already logged, a corrected step 6
# needs additional room.

# allow only:
# current logged calls
# + exact step-6 calls still required
# + 2 calls of safety headroom.

current_logged_calls = calls()

corrected_step6_cap = (
    current_logged_calls
    +
    needed_calls
    +
    2
)

if MAX_TOTAL_LLM_CALLS < corrected_step6_cap:

    print(
        f'Adjusting hard cap for corrected rerun: '
        f'{MAX_TOTAL_LLM_CALLS} -> '
        f'{corrected_step6_cap}'
    )

    MAX_TOTAL_LLM_CALLS = (
        corrected_step6_cap
    )

print()
print(
    "========== CORRECTED STEP-6 PREVIEW =========="
)

print(
    f'Already complete:       '
    f'{len(done):,} / {len(h):,}'
)

print(
    f'Pairs still required:   '
    f'{len(pend):,}'
)

print(
    f'Batch size:             '
    f'{CONSTITUTION_BATCH_SIZE}'
)

print(
    f'Expected new calls:     '
    f'{needed_calls}'
)

print(
    'Automatic retries:     0'
)

print(
    f'Current logged calls:   '
    f'{current_logged_calls}'
)

print(
    f'Corrected hard cap:     '
    f'{MAX_TOTAL_LLM_CALLS}'
)

print(
    "================================================"
)

# response format

# critical:

# instead of showing only:
# c5_01
# c10_01
# c20_01

# we request arrays with exact known lengths.

# the array positions correspond exactly to the id lists above.

response_schema = {

    "results": [

        {
            "pair_id":
                "...",

            "full_votes": {
                "5": "A|B",
                "10": "A|B",
                "20": "A|B"
            },

            "full_confidence": {
                "5": 0.0,
                "10": 0.0,
                "20": 0.0
            },

            "principle_votes": {

                "5": [
                    "A|B|None"
                ] * 5,

                "10": [
                    "A|B|None"
                ] * 10,

                "20": [
                    "A|B|None"
                ] * 20
            }
        }

    ]
}

# run corrected evaluation

bad_pair_ids = []

for batch in tqdm(

    list(
        chunks(
            pend,
            CONSTITUTION_BATCH_SIZE
        )
    ),

    desc='CORRECTED 5/10/20 constitution evaluation'

):

    user = f'''
Evaluate THREE INDEPENDENT r/AskScience constitutions on
EVERY supplied pair.

For EACH pair and EACH constitution size (5, 10, and 20):

TASK 1 — COMPLETE CONSTITUTION
Use the COMPLETE constitution of that size to predict which
post the r/AskScience community would prefer.

Return:
- "A" or "B"
- confidence from 0 to 1

TASK 2 — EVERY INDIVIDUAL CONSTITUENT PRINCIPLE
Apply EACH constituent principle independently to the same
pair.

For each principle return:
- "A" if Post A better satisfies that principle
- "B" if Post B better satisfies that principle
- "None" only if that individual principle is genuinely
  inapplicable or does not distinguish the two posts

CRITICAL OUTPUT REQUIREMENT:

You MUST evaluate EVERY principle.

The principle vote arrays MUST correspond EXACTLY to these
principle-ID orders:

{json.dumps(required_constitution_ids, ensure_ascii=False)}

Therefore:

- principle_votes["5"] MUST contain exactly 5 votes
- principle_votes["10"] MUST contain exactly 10 votes
- principle_votes["20"] MUST contain exactly 20 votes

Do NOT omit any vote.
Do NOT shorten an array.
Do NOT return only the first principle.
Do NOT substitute missing values.
Every position must contain exactly one of:

"A"
"B"
"None"

The first vote in each array belongs to the first ID in the
corresponding ID list above, the second vote belongs to the
second ID, and so on.

Return JSON ONLY in exactly this structure:

{json.dumps(response_schema)}

CONSTITUTIONS:

{json.dumps(constitution_payload, ensure_ascii=False)}

PAIRS:

{json.dumps(batch, ensure_ascii=False)}
'''.strip()

    out = ask(

        (
            'Evaluate all three complete constitutions and '
            'EVERY constituent principle independently. '
            'Return all required votes with no omissions. '
            'Return JSON only.'
        ),

        user,

        'constitution_bundle_test_CORRECTED',

        retries=1,

        max_tokens=16000

    )

    got = {

        str(x.get('pair_id')):
            x

        for x in out.get(
            'results',
            []
        )

        if isinstance(
            x,
            dict
        )

    }

    valid_rows = []

    batch_bad = []

    # strict parser

    for x in batch:

        pid = str(
            x['pair_id']
        )

        # entire pair missing.
        if pid not in got:

            batch_bad.append(
                pid
            )

            continue

        g = got[
            pid
        ]

        fv = g.get(
            'full_votes'
        )

        fc = g.get(
            'full_confidence'
        )

        pvall = g.get(
            'principle_votes'
        )

        if not isinstance(
            fv,
            dict
        ):

            batch_bad.append(
                pid
            )

            continue

        if not isinstance(
            fc,
            dict
        ):

            fc = {}

        if not isinstance(
            pvall,
            dict
        ):

            batch_bad.append(
                pid
            )

            continue

        # validate complete-constitution a/b votes.

        clean_full_votes = {}

        full_vote_valid = True

        for size in FINAL_CONSTITUTION_SIZES:

            sk = str(
                size
            )

            vote = str(
                fv.get(
                    sk,
                    ''
                )
            ).strip()

            if vote not in {
                'A',
                'B'
            }:

                full_vote_valid = False

                break

            clean_full_votes[
                sk
            ] = vote

        if not full_vote_valid:

            batch_bad.append(
                pid
            )

            continue

        # confidence values

        clean_confidence = {}

        for size in FINAL_CONSTITUTION_SIZES:

            sk = str(
                size
            )

            try:

                conf = float(
                    fc.get(
                        sk,
                        np.nan
                    )
                )

            except Exception:

                conf = np.nan

            clean_confidence[
                sk
            ] = conf

        # strict constituent-principle validation.

        # missing vote != none.

        # if any required vote is absent/invalid, this pair
        # is not checkpointed.

        clean_pv = {}

        principles_valid = True

        for size in FINAL_CONSTITUTION_SIZES:

            sk = str(
                size
            )

            raw_votes = pvall.get(
                sk
            )

            if not isinstance(
                raw_votes,
                list
            ):

                principles_valid = False

                break

            required_ids = (
                required_constitution_ids[
                    sk
                ]
            )

            # exact number required.
            if len(
                raw_votes
            ) != len(
                required_ids
            ):

                principles_valid = False

                break

            normalized_votes = [

                str(v).strip()

                for v in raw_votes

            ]

            if any(

                v not in {
                    'A',
                    'B',
                    'None'
                }

                for v
                in normalized_votes

            ):

                principles_valid = False

                break

            clean_pv[
                sk
            ] = {

                cid:
                    vote

                for cid, vote
                in zip(
                    required_ids,
                    normalized_votes
                )

            }

        if not principles_valid:

            batch_bad.append(
                pid
            )

            continue

        # only fully valid pairs are saved.

        valid_rows.append({

            'pair_id':
                pid,

            'full_votes':
                clean_full_votes,

            'full_confidence':
                clean_confidence,

            'principle_votes':
                clean_pv

        })

    # checkpoint valid results immediately

    if valid_rows:

        append_jsonl(
            ckc,
            valid_rows
        )

    bad_pair_ids.extend(
        batch_bad
    )

    print(
        f'\nBatch returned '
        f'{len(valid_rows):,}/{len(batch):,} '
        f'fully valid pairs; '
        f'{len(batch_bad):,} incomplete.'
    )

# completeness check

raw_const = read_jsonl(
    ckc
)

completed_ids = {

    str(r['pair_id'])

    for r in raw_const

}

expected_ids = set(
    h.pair_id.astype(str)
)

remaining_ids = (
    expected_ids
    -
    completed_ids
)

print()
print(
    "========== CORRECTED STEP-6 COMPLETENESS =========="
)

print(
    f'Fully valid pairs: '
    f'{len(completed_ids):,} / {len(h):,}'
)

print(
    f'Still missing:     '
    f'{len(remaining_ids):,}'
)

print(
    "===================================================="
)

if remaining_ids:

    print()

    print(
        'IMPORTANT: Missing pairs were NOT converted to None.'
    )

    print(
        'Rerun THIS SAME CELL later and it will request '
        'only the remaining pairs.'
    )

    print()

    print(
        'First missing IDs:',
        sorted(
            remaining_ids
        )[:20]
    )

    raise RuntimeError(

        f'Corrected Step 6 is not complete yet: '
        f'{len(remaining_ids):,} held-out pairs remain. '
        'No graphs should be generated until this reaches zero.'

    )

# build corrected metrics

truth = (

    h.set_index(
        'pair_id'
    )

    .preferred_position

    .to_dict()

)

ce_by_size = {}

constitution_metrics_by_size = {}

result_rows = []

for size in FINAL_CONSTITUTION_SIZES:

    sk = str(
        size
    )

    full_rows = []

    pvrows = []

    for r0 in raw_const:

        pid = str(
            r0[
                'pair_id'
            ]
        )

        full_rows.append({

            'pair_id':
                pid,

            'vote':
                str(
                    r0[
                        'full_votes'
                    ][
                        sk
                    ]
                ),

            'confidence':
                r0[
                    'full_confidence'
                ].get(
                    sk,
                    np.nan
                )

        })

        # every principle is guaranteed to exist here.
        for cid in required_constitution_ids[
            sk
        ]:

            pvrows.append({

                'selected_id':
                    str(
                        cid
                    ),

                'pair_id':
                    pid,

                'vote':
                    str(
                        r0[
                            'principle_votes'
                        ][
                            sk
                        ][
                            cid
                        ]
                    )

            })

    # full constitution accuracy

    ce = pd.DataFrame(
        full_rows
    )

    if len(ce) != len(h):

        raise RuntimeError(
            f'{size}-principle full constitution has '
            f'{len(ce):,} predictions; '
            f'expected {len(h):,}.'
        )

    ce[
        'truth'
    ] = ce[
        'pair_id'
    ].map(
        truth
    )

    ce[
        'correct'
    ] = ce[
        'vote'
    ].eq(
        ce[
            'truth'
        ]
    )

    ce_by_size[
        size
    ] = ce

    ca = ce[
        'correct'
    ].mean()

    clo, chi = boot(
        ce[
            'correct'
        ]
    )

    uplift = (
        ca
        -
        no_acc
    )

    result_rows.append({

        'constitution_size':
            size,

        'accuracy':
            ca,

        'ci_low':
            clo,

        'ci_high':
            chi,

        'uplift_vs_no_constitution':
            uplift

    })

    # individual constituent principle metrics

    cpv = pd.DataFrame(
        pvrows
    )

    expected_constituent_votes = (
        len(h)
        *
        size
    )

    if len(cpv) != expected_constituent_votes:

        raise RuntimeError(

            f'{size}-principle constitution has '
            f'{len(cpv):,} constituent votes; '
            f'expected {expected_constituent_votes:,}.'

        )

    # every constituent should have exactly 5,000 judgments.
    per_principle_counts = (

        cpv

        .groupby(
            'selected_id'
        )

        .pair_id

        .nunique()

    )

    if (
        len(
            per_principle_counts
        ) != size

        or

        per_principle_counts.min()
        != len(h)

        or

        per_principle_counts.max()
        != len(h)
    ):

        raise RuntimeError(

            f'{size}-principle constituent coverage '
            'is incomplete.'

        )

    cpv[
        'truth'
    ] = cpv[
        'pair_id'
    ].map(
        truth
    )

    cpv[
        'applicable'
    ] = cpv[
        'vote'
    ].isin(
        [
            'A',
            'B'
        ]
    )

    cpv[
        'correct'
    ] = (

        cpv[
            'applicable'
        ]

        &

        cpv[
            'vote'
        ].eq(
            cpv[
                'truth'
            ]
        )

    )

    cm = (

        cpv

        .groupby(
            'selected_id'
        )

        .agg(

            tested=(
                'pair_id',
                'nunique'
            ),

            applicable=(
                'applicable',
                'sum'
            ),

            correct=(
                'correct',
                'sum'
            )

        )

        .reset_index()

    )

    cm[
        'applicability_rate'
    ] = (

        cm[
            'applicable'
        ]

        /

        cm[
            'tested'
        ]

    )

    cm[
        'accuracy_when_applicable'
    ] = (

        cm[
            'correct'
        ]

        /

        cm[
            'applicable'
        ].replace(
            0,
            np.nan
        )

    )

    cm[
        'incorrect'
    ] = (

        cm[
            'applicable'
        ]

        -
        cm[
            'correct'
        ]

    )

    cm[
        'net_score'
    ] = (

        cm[
            'correct'
        ]

        -
        cm[
            'incorrect'
        ]

    )

    # add constitution text back in.
    cm = (

        constitutions[
            size
        ]

        .rename(
            columns={
                'constitution_id':
                    'selected_id'
            }
        )

        .merge(
            cm,
            on='selected_id',
            how='left',
            validate='one_to_one'
        )

    )

    constitution_metrics_by_size[
        size
    ] = cm

    # save corrected files

    cm.to_csv(

        OUTPUT_DIR /
        f'constitution_{size}_principle_metrics.csv',

        index=False

    )

    cpv.to_csv(

        OUTPUT_DIR /
        f'constitution_{size}_principle_votes.csv',

        index=False

    )

    ce.to_csv(

        OUTPUT_DIR /
        f'constitution_{size}_full_votes.csv',

        index=False

    )

# size results

constitution_results = pd.DataFrame(
    result_rows
)

constitution_results.to_csv(

    OUTPUT_DIR /
    'constitution_results_5_10_20.csv',

    index=False

)

# rebuild baseline table for downstream report

constitution_baseline_rows = []

for _, r0 in (
    constitution_results
    .iterrows()
):

    size = int(
        r0[
            'constitution_size'
        ]
    )

    constitution_baseline_rows.append({

        'method':
            f'Final inferred constitution '
            f'({size} principles)',

        'accuracy':
            float(
                r0[
                    'accuracy'
                ]
            ),

        'ci_low':
            float(
                r0[
                    'ci_low'
                ]
            ),

        'ci_high':
            float(
                r0[
                    'ci_high'
                ]
            ),

        'notes':
            (
                f'Independent '
                f'{size}-principle constitution'
            )

    })

final_baseline = pd.concat(

    [
        baseline,
        pd.DataFrame(
            constitution_baseline_rows
        )
    ],

    ignore_index=True

)

final_baseline.to_csv(

    OUTPUT_DIR /
    'baseline_comparison_final.csv',

    index=False

)

# final audit

print()
print(
    "========== CORRECTED STEP-6 RESULT =========="
)

print(
    f'Held-out pairs: '
    f'{len(h):,}'
)

print()

for size in FINAL_CONSTITUTION_SIZES:

    cm = (
        constitution_metrics_by_size[
            size
        ]
    )

    cr = (
        constitution_results[
            constitution_results[
                'constitution_size'
            ].eq(
                size
            )
        ]
        .iloc[
            0
        ]
    )

    print(
        f'{size}-PRINCIPLE CONSTITUTION'
    )

    print(
        f'  full accuracy: '
        f'{cr.accuracy:.2%}'
    )

    print(
        f'  constituent principles: '
        f'{len(cm)} / {size}'
    )

    print(
        f'  tested per constituent: '
        f'{cm.tested.min():,.0f}–'
        f'{cm.tested.max():,.0f}'
    )

    print(
        f'  constituents with >0 applicable votes: '
        f'{int((cm.applicable > 0).sum())} / {size}'
    )

    print()

print(
    'Saved corrected metric files for '
    '5, 10, and 20 principles.'
)

print(
    'You can now rerun the EXISTING '
    '5/10/20 graph cells unchanged.'
)

print(
    "============================================="
)


In [ ]:
# this was the next smaller repair batch.
CONSTITUTION_BATCH_SIZE = 15

print("Repair batch size:", CONSTITUTION_BATCH_SIZE)
print("Existing 4,942 valid results will be reused.")
print("Expected calls for 58 remaining pairs: 4")


In [ ]:
CONSTITUTION_BATCH_SIZE = 4

print("Final repair batch size:", CONSTITUTION_BATCH_SIZE)
print("Existing 4,992 valid results will be reused.")
print("Expected calls for 8 remaining pairs: 2")


### 5-principle constitution — accuracy and applicability


In [ ]:
ROOT_OUTPUT_DIR = OUTPUT_DIR
metrics = constitution_metrics_by_size[5].copy()

OUTPUT_DIR = ROOT_OUTPUT_DIR / 'constitution_5'
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

p = metrics.sort_values(
    'accuracy_when_applicable',
    ascending=False
)

x = np.arange(len(p))
w = .38

plt.figure(figsize=(10, 5))
plt.bar(
    x - w / 2,
    p.accuracy_when_applicable,
    w,
    label='Accuracy'
)
plt.bar(
    x + w / 2,
    p.applicability_rate,
    w,
    label='Applicability'
)
plt.axhline(.5, linestyle='--')
plt.xticks(x, p.selected_id)
plt.ylim(0, 1)
plt.legend()
plt.title('Principle accuracy and applicability')
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / 'principle_accuracy_and_applicability.png',
    dpi=180
)
plt.close()

OUTPUT_DIR = ROOT_OUTPUT_DIR


### 10-principle constitution — accuracy and applicability


In [ ]:
ROOT_OUTPUT_DIR = OUTPUT_DIR
metrics = constitution_metrics_by_size[10].copy()

OUTPUT_DIR = ROOT_OUTPUT_DIR / 'constitution_10'
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

p = metrics.sort_values(
    'accuracy_when_applicable',
    ascending=False
)

x = np.arange(len(p))
w = .38

plt.figure(figsize=(10, 5))
plt.bar(
    x - w / 2,
    p.accuracy_when_applicable,
    w,
    label='Accuracy'
)
plt.bar(
    x + w / 2,
    p.applicability_rate,
    w,
    label='Applicability'
)
plt.axhline(.5, linestyle='--')
plt.xticks(x, p.selected_id)
plt.ylim(0, 1)
plt.legend()
plt.title('Principle accuracy and applicability')
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / 'principle_accuracy_and_applicability.png',
    dpi=180
)
plt.close()

OUTPUT_DIR = ROOT_OUTPUT_DIR


### 20-principle constitution — accuracy and applicability


In [ ]:
ROOT_OUTPUT_DIR = OUTPUT_DIR
metrics = constitution_metrics_by_size[20].copy()

OUTPUT_DIR = ROOT_OUTPUT_DIR / 'constitution_20'
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

p = metrics.sort_values(
    'accuracy_when_applicable',
    ascending=False
)

x = np.arange(len(p))
w = .38

plt.figure(figsize=(10, 5))
plt.bar(
    x - w / 2,
    p.accuracy_when_applicable,
    w,
    label='Accuracy'
)
plt.bar(
    x + w / 2,
    p.applicability_rate,
    w,
    label='Applicability'
)
plt.axhline(.5, linestyle='--')
plt.xticks(x, p.selected_id)
plt.ylim(0, 1)
plt.legend()
plt.title('Principle accuracy and applicability')
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / 'principle_accuracy_and_applicability.png',
    dpi=180
)
plt.close()

OUTPUT_DIR = ROOT_OUTPUT_DIR


## step 7 — compare the inferred constitutions with the official AskScience rules


In [ ]:
rf = OUTPUT_DIR / 'step7_rules_comparison_5_10_20.json'

if rf.exists() and not FORCE_RULES_COMPARISON:
    rules = load_json(rf)

else:
    budget(
        'Rules comparison for 5/10/20 constitutions',
        1
    )

    payload = {
        str(s): constitutions[s].to_dict('records')
        for s in FINAL_CONSTITUTION_SIZES
    }

    rules = ask(
        'Compare inferred behavioral principles with explicit moderation rules. Return JSON only.',
        f'''Official rules: {json.dumps(OFFICIAL_RULES,ensure_ascii=False)}. Independent inferred constitutions by size: {json.dumps(payload,ensure_ascii=False)}. Compare each constitution separately, then summarize how the comparison changes with constitution size. Return {{"overall_comparison":"...","by_size":{{"5":{{"summary":"...","principle_mappings":[{{"constitution_id":"C5_01","relationship":"aligns|extends|unrelated|tension","matched_rule_numbers":[1],"analysis":"..."}}],"official_rules_not_reflected":["..."],"inferred_preferences_not_in_rules":["..."]}},"10":{{"summary":"..."}},"20":{{"summary":"..."}}}},"size_comparison":"..."}}.''',
        'rules_comparison_5_10_20'
    )

    save_json(
        rf,
        rules
    )

print(
    rules.get(
        'overall_comparison',
        ''
    )
)

print(
    rules.get(
        'size_comparison',
        ''
    )
)


## step 8 — look at common errors and possible biases


In [ ]:
# this pulls the highest-confidence mistakes so i can compare failure patterns across sizes.
reuse = Counter(
    pd.concat([
        h.preferred_post_id.astype(str),
        h.nonpreferred_post_id.astype(str)
    ])
)

summaries = {}
samples = {}
err_by_size = {}

for size in FINAL_CONSTITUTION_SIZES:
    ce = ce_by_size[size].copy()

    err = ce.merge(
        h,
        on='pair_id'
    )

    err = err[
        ~err.correct
    ].copy()

    err['confidence'] = pd.to_numeric(
        err.confidence,
        errors='coerce'
    )

    err = err.sort_values(
        'confidence',
        ascending=False
    )

    err_by_size[size] = err

    rr = constitution_results[
        constitution_results.constitution_size == size
    ].iloc[0]

    summaries[str(size)] = {
        'n_pairs': len(h),
        'constitution_accuracy': float(rr.accuracy),
        'no_constitution_accuracy': float(no_acc),
        'uplift': float(rr.uplift_vs_no_constitution)
    }

    samples[str(size)] = [
        {
            'pair_id': x.pair_id,
            'truth': x.truth,
            'model_vote': x.vote,
            'confidence': (
                None
                if pd.isna(x.confidence)
                else float(x.confidence)
            ),
            'post_a': x.post_a,
            'post_b': x.post_b
        }
        for _, x in err.head(10).iterrows()
    ]

q = {
    'by_size': summaries,
    'pair_distance_vs_upvote_gap_spearman': sr,
    'repeated_posts': sum(
        v > 1
        for v in reuse.values()
    ),
    'max_reuse': (
        max(reuse.values())
        if reuse
        else 0
    )
}

ef = OUTPUT_DIR / 'step8_error_analysis_5_10_20.json'

if ef.exists() and not FORCE_ERROR_ANALYSIS:
    ea = load_json(ef)

else:
    budget(
        'Error analysis across 5/10/20',
        1
    )

    constitutions_text = {
        str(s):
            constitutions[s][
                ['constitution_id', 'principle']
            ].to_dict('records')
        for s in FINAL_CONSTITUTION_SIZES
    }

    ea = ask(
        'Analyze model failures without claiming causal voter motives. Return JSON only.',
        f'''Independent constitutions: {json.dumps(constitutions_text,ensure_ascii=False)}. Quantitative summary: {json.dumps(q)}. High-confidence error samples by constitution size: {json.dumps(samples,ensure_ascii=False)}. Compare recurring failure modes across sizes and identify whether added principles appear to help, hurt, or leave errors unchanged without making causal claims. Return {{"recurring_failure_modes":[{{"mode":"...","sizes_affected":[5,10,20],"evidence":"...","severity":"high|medium|low"}}],"possible_surface_biases":[{{"bias":"...","sizes_affected":[5,10,20],"evidence":"...","caution":"..."}}],"data_quality_concerns":["..."],"size_tradeoffs":[{{"comparison":"5 vs 10|10 vs 20|5 vs 20","observation":"..."}}],"recommended_next_checks":["..."],"interpretation_caveat":"..."}}.''',
        'error_analysis_5_10_20'
    )

    save_json(
        ef,
        ea
    )


## step 9 — export the final workbook, figures, and report


In [ ]:
# this recreates the baseline plot before the report tries to use it.
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.barh(
    final_baseline.method,
    final_baseline.accuracy
)
plt.axvline(
    0.5,
    linestyle='--'
)
plt.xlim(0, 1)
plt.title('5/10/20 constitutions vs baselines')
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / 'baseline_comparison.png',
    dpi=180
)

plt.close()

print(
    'Created:',
    OUTPUT_DIR / 'baseline_comparison.png'
)
print('PAID LLM CALLS MADE: 0')


In [ ]:
# this writes the final workbook and report after all of the plots are saved.
workbook = (
    OUTPUT_DIR
    / 'AskScience_ICAI_final_validation_5_10_20_results.xlsx'
)

with pd.ExcelWriter(
    workbook,
    engine='openpyxl'
) as w:
    initial_20.to_excel(
        w,
        sheet_name='Initial 20',
        index=False
    )

    review_df.to_excel(
        w,
        sheet_name='LLM Review',
        index=False
    )

    selected_df.to_excel(
        w,
        sheet_name='Selected 20',
        index=False
    )

    embedding_metrics.to_excel(
        w,
        sheet_name='Embedding Metrics',
        index=False
    )

    selected_metrics.to_excel(
        w,
        sheet_name='Selected Metrics',
        index=False
    )

    constitution_results.to_excel(
        w,
        sheet_name='Size Results',
        index=False
    )

    final_baseline.to_excel(
        w,
        sheet_name='Baselines',
        index=False
    )

    for size in FINAL_CONSTITUTION_SIZES:
        constitutions[size].to_excel(
            w,
            sheet_name=f'Constitution {size}',
            index=False
        )

        constitution_metrics_by_size[size].to_excel(
            w,
            sheet_name=f'Const Metrics {size}',
            index=False
        )

        err_by_size[size].head(100).to_excel(
            w,
            sheet_name=f'Top Errors {size}',
            index=False
        )

    pd.DataFrame(
        read_jsonl(CALL_LOG)
    ).to_excel(
        w,
        sheet_name='LLM Calls',
        index=False
    )

report = Document()
report.add_heading(
    'ICAI Held-Out Validation: Independent 5/10/20 Constitutions',
    0
)

p = report.add_paragraph(
    'r/AskScience preference-pair study'
)
p.alignment = WD_ALIGN_PARAGRAPH.CENTER

report.add_heading(
    'Executive summary',
    1
)

parts = []

for _, r0 in constitution_results.iterrows():
    parts.append(
        f"{int(r0.constitution_size)} principles: "
        f"{r0.accuracy:.1%} accuracy "
        f"(95% CI {r0.ci_low:.1%}-{r0.ci_high:.1%}), "
        f"uplift {r0.uplift_vs_no_constitution:+.1%}"
    )

report.add_paragraph(
    f'Exactly {len(h):,} held-out pairs were used. '
    'Twenty candidate values were tested on all held-out pairs. '
    'Three independent constitutions were generated in one LLM call. '
    f'No-constitution baseline: {no_acc:.1%}. '
    + '; '.join(parts)
    + f'. Total Azure calls: {calls()}.'
)

report.add_heading(
    '1. Selected-principle accuracy and stability',
    1
)

for img in [
    'principle_accuracy_and_applicability.png',
    'principle_cumulative_accuracy.png',
    'embedding_vs_llm_accuracy.png'
]:
    pth = OUTPUT_DIR / img

    if pth.exists():
        report.add_picture(
            str(pth),
            width=Inches(6.2)
        )

report.add_heading(
    '2. Final-constitution principle accuracy and applicability',
    1
)

for size in FINAL_CONSTITUTION_SIZES:
    report.add_heading(
        f'{size}-principle constitution',
        2
    )

    pth = (
        OUTPUT_DIR
        / f'constitution_{size}'
        / 'principle_accuracy_and_applicability.png'
    )

    if pth.exists():
        report.add_picture(
            str(pth),
            width=Inches(6.2)
        )

report.add_heading(
    '3. Embedding correlation diagnostics',
    1
)

report.add_paragraph(
    'Pair semantic distance vs upvote-ratio gap: '
    f'Pearson r={pr:.3f}; Spearman rho={sr:.3f}.'
)

report.add_picture(
    str(
        OUTPUT_DIR
        / 'embedding_distance_vs_upvote_gap.png'
    ),
    width=Inches(6.2)
)

report.add_heading(
    '4. Independent final constitutions',
    1
)

for size in FINAL_CONSTITUTION_SIZES:
    report.add_heading(
        f'{size}-principle constitution',
        2
    )

    for _, x in constitutions[size].iterrows():
        report.add_paragraph(
            x.principle,
            style='List Number'
        )

report.add_heading(
    '5. Baselines and constitution-size comparison',
    1
)

report.add_picture(
    str(
        OUTPUT_DIR
        / 'baseline_comparison.png'
    ),
    width=Inches(6.2)
)

report.add_heading(
    '6. Official rules comparison',
    1
)

report.add_paragraph(
    rules.get(
        'overall_comparison',
        ''
    )
)

report.add_paragraph(
    rules.get(
        'size_comparison',
        ''
    )
)

report.add_heading(
    '7. Error and bias analysis',
    1
)

for x in ea.get(
    'recurring_failure_modes',
    []
):
    report.add_paragraph(
        f"{x.get('mode','')}: {x.get('evidence','')}",
        style='List Bullet'
    )

report.add_heading(
    '8. Cost/reproducibility',
    1
)

report.add_paragraph(
    'All selected-principle votes and the no-constitution baseline '
    'share one held-out pass. All 5/10/20 full-constitution and '
    'constituent-principle votes share a second held-out pass. '
    'The three constitutions are generated together in one LLM call. '
    'Paid calls are blocked by default, the full fresh-run plan is '
    'about 121 calls, and the emergency hard cap is 130. '
    'Every successful Azure call is logged with token counts when '
    f'available. Total calls: {calls()}.'
)

report.add_heading(
    '9. Limitations',
    1
)

for x in [
    'LLM judgments remain model-dependent even when calls are batched efficiently.',
    'Upvote ratio is observational and can reflect visibility, timing, topic interest, or platform effects.',
    'The three inferred constitutions are independent lossy compressions of observed preferences and are not equivalent to official moderation policy.',
    'Constitution-size comparisons should be interpreted as predictive comparisons on this held-out sample, not causal effects of adding principles.'
]:
    report.add_paragraph(
        x,
        style='List Bullet'
    )

report_path = (
    OUTPUT_DIR
    / 'AskScience_ICAI_final_validation_5_10_20_report.docx'
)

report.save(report_path)

print(report_path)
print(workbook)
print('Total calls:', calls())
